## Step 1: Download and load the Constitution of Nepal (2015) PDF

We use the officially-based English translation (as amended through 2020,
the current legally in-force version as of this project). Note: a further
constitutional amendment was under discussion in the Nepali government as
of 2026, which may not be reflected in this text.

In [4]:
!wget -O nepal_constitution.pdf "https://www.ecoi.net/en/file/local/1125402/1930_1444821984_561625364.pdf"

from pypdf import PdfReader

reader = PdfReader("nepal_constitution.pdf")
print("Number of pages:", len(reader.pages))
print("\nFirst page text preview:")
print(reader.pages[0].extract_text()[:500])

--2026-09-12 09:47:21--  https://www.ecoi.net/en/file/local/1125402/1930_1444821984_561625364.pdf
Resolving www.ecoi.net (www.ecoi.net)... 188.34.190.110, 2a01:4f8:c17:52d7::1
Connecting to www.ecoi.net (www.ecoi.net)|188.34.190.110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4937095 (4.7M) [application/pdf]
Saving to: ‘nepal_constitution.pdf’

nepal_constitution. 100%[===================>]   4.71M  4.35MB/s    in 1.1s    

2026-09-12 09:47:23 (4.35 MB/s) - ‘nepal_constitution.pdf’ saved [4937095/4937095]

Number of pages: 161

First page text preview:
Unofficial translation 
 
 
CONSTITUTION OF NEPAL 2015 
 
 
 
 
 
 
 
 
 
 
 
Constituent Assembly Secretariat 
Singha Durbar 
 
UNOFFICIAL TRANSLATION BY 
 
 
 



## Step 2: Extract full text from the PDF

Now that we've confirmed the PDF loads correctly, extract text from every
page and inspect the total length and quality of extraction — PDF text
extraction can sometimes produce messy output (broken words, extra spaces),
so it's worth checking before building the RAG pipeline on top of it.

In [5]:
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print("Total characters extracted:", len(full_text))
print("\nSample from the middle of the document:")
print(full_text[20000:21000])

Total characters extracted: 340835

Sample from the middle of the document:
 item, editorial, article, feature, or other reading material, or the 
use of audio-visual material by any medium, including electronic publication, 
broadcasting and printing.   
Provided that nothing shall be deemed to prevent the making of laws to 
impose reasonable restriction on any act which may undermine the 
nationality, sovereignty, and indivisibility of Nepal, or the good relations 
between federal units, or jeopardizes the harmonious relations subsisting 
among different caste groups and tribes, or communities, or an act of 
treason, or defamation of social dignity of individuals through the publication 
and dissemination of false material, or  contemp t of court, or material that 
incites criminal offence, or an act that is contrary to decent public behavior 
and morality, or disrespects labor, or incites untouchability or gender 
discriminations.   
(2) If there is any broadcasting, publishing or p

## Step 3: Chunk the text for retrieval

LLMs and embedding models work best on smaller pieces of text, not one
giant 340K-character blob. We split the document into overlapping chunks
(e.g. 500 characters each, with 50 characters of overlap) so that context
isn't awkwardly cut off at chunk boundaries — a sentence split across two
chunks would lose meaning without some overlap.

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text)
print("Number of chunks:", len(chunks))
print("\nExample chunk:")
print(chunks[50])

Number of chunks: 758

Example chunk:
eriod of twenty -four hours after such arrest, 
excluding the time n ecessary for the journey from the time and place 
of arrest to such authority, and the arrested person shall not be 
detained in custody beyond the said period except on the order of such 
authority.      
Provided that this clause shall not apply to a person in preventive detention or to a 
citizen of an enemy state. 
(4) No person shall be punished for an act which was not punishable by law 
when the act was committed, and no


## Step 4: Embed chunks and build a FAISS vector index

Each text chunk gets converted into a numerical vector (embedding) using a
pretrained sentence-embedding model — this captures the *meaning* of the
text, not just keywords. We use `sentence-transformers`, a well-established
library for this.

All embeddings are then stored in a FAISS index, which allows extremely
fast similarity search: given a question's embedding, FAISS finds the
chunks whose embeddings are closest in meaning.

In [7]:
!pip install sentence-transformers faiss-cpu -q

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding all chunks... this may take a minute")
embeddings = embed_model.encode(chunks, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)

# build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print("Number of vectors in index:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.1 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding all chunks... this may take a minute


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embeddings shape: (758, 384)
Number of vectors in index: 758


## Step 5: Retrieve relevant chunks for a query

Given a question, we embed it using the same model, then search the FAISS
index for the chunks whose embeddings are closest (most similar in meaning).
This is the "Retrieval" part of RAG.

In [8]:
def retrieve(query, k=3):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k)
    results = [chunks[i] for i in indices[0]]
    return results

# test it
query = "What are the fundamental rights of citizens?"
results = retrieve(query, k=3)

for i, r in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(r)
    print()

--- Result 1 ---
 Right to live with dignity : (1) Each person shall have the rig ht to live with 
dignity. 
(2) No law shall be made for capital punishment. 
17. Right to Freedom : (1) Except as provided for by law no person shall be 
deprived of her/his personal liberty.  
(2) Every citizen shall have the following freedoms:  
(a) freedom of opinion and expression, 
(b) freedom to assemble peacefully and without arms, 
(c) freedom to form political party, 
(d) freedom to form unions and associations, 
(e) free

--- Result 2 ---

- 15 - 
(2) A person who has suffered from sub-standard object or service shall have 
the right to be compensated as provided for by law.  
45. Right against exile: No citizen shall be exiled. 
46. Right to constitutional r emedy: There shall be right to constitutional remedy 
pursuant to  the Articles 133 or 144 in course of implementation of rights 
granted in this part. 
47. Implementation of fundamental rights : For the enforcement of the rights 
conferre

## Step 6: Generate an answer using retrieved context (the "Generation" in RAG)

Now we combine retrieval with generation: take the user's question, retrieve
the most relevant chunks, then pass both the question and the retrieved
text to an LLM, instructing it to answer using only that context. This
grounds the answer in the actual constitution text rather than the LLM's
general training knowledge.

We use Google's Gemini API for generation.

In [11]:
!pip install -q google-genai

from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

def generate_answer(query, k=3):
    retrieved_chunks = retrieve(query, k=k)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question using ONLY the context below, which is from the Constitution of Nepal (2015, as amended). If the context doesn't contain enough information to answer, say so clearly rather than guessing.

Context:
{context}

Question: {query}

Answer:"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
    )
    return response.text

answer = generate_answer("What are the fundamental rights of citizens?")
print(answer)

Based on the provided context, the following rights are explicitly mentioned:

*   **Right to live with dignity:** Each person has the right to live with dignity, and no law shall be made for capital punishment.
*   **Right to Freedom:** Except as provided by law, no person shall be deprived of personal liberty. Citizens specifically have the freedom of:
    *   Opinion and expression
    *   Peaceful assembly without arms
    *   Forming a political party
    *   Forming unions and associations
*   **Right to compensation:** A person who suffers from a sub-standard object or service has the right to be compensated by law.
*   **Right against exile:** No citizen shall be exiled.
*   **Right to constitutional remedy:** The right to a constitutional remedy pursuant to Articles 133 or 144.
*   **Health and Hygiene Rights:** 
    *   No citizen shall be deprived of emergency health care.
    *   Right to be informed about one's health condition regarding health care services.
    *   Equal

In [12]:
test_questions = [
    "What is the process for amending the constitution?",
    "What are the qualifications to become Prime Minister of Nepal?",
    "What does the constitution say about the death penalty?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(generate_answer(q))
    print("\n" + "="*80 + "\n")

Q: What is the process for amending the constitution?
Based on the provided context, the process and conditions for amending the constitution include:

* **Limitations:** The constitution cannot be amended in any way that contravenes Nepal's self-rule, sovereignty, territorial integrity, and sovereignty vested in the people.
* **Proposal:** Except for the restricted matters mentioned above, if an amendment is sought regarding matters that fall under the fundamentals of the constitution, the proposal must be presented to either house of the federal legislature.

*Note: The provided text cuts off mid-sentence ("Provided that, Claus"), so the context does not contain the complete information regarding the entire amendment process.*


Q: What are the qualifications to become Prime Minister of Nepal?
Based on the provided context, there is no information about the qualifications to become the Prime Minister of Nepal. The context only outlines the qualifications for the President, local muni

In [13]:
answer = generate_answer("What are the qualifications to become Prime Minister of Nepal?", k=7)
print(answer)

Based on the provided context, there is no information about the qualifications required to become the Prime Minister of Nepal.


In [14]:
import re

# find all chunks that mention "Prime Minister"
pm_chunks = [c for c in chunks if "Prime Minister" in c]
print(f"Chunks mentioning 'Prime Minister': {len(pm_chunks)}")
print("\nFirst few:")
for c in pm_chunks[:3]:
    print(c)
    print("---")


Chunks mentioning 'Prime Minister': 40

First few:
es of the State and gradually implementing the  
policies.  
53. Submitting report:  An annual report regarding the works of the government 
including the achievements made in the implementation of the directive 
Constitution of Nepal 2015, Unofficial English Translation 
 
- 25 - 
principles, policies and responsibilities mentioned in this Part, shall be 
presented to the President. The President shall make arrangements to send 
such reports to the Federal Legislature through the Prime Minister
---
the Federal Legislature through the Prime Minister. 
54 .Provision regarding monitoring:  (1) There shall be a committee in the 
Parliament as provided for in law to monitor the progressiv e implementation 
of the directive principles, policies and responsibilities of the state as 
mentioned in this Part.  
55. Questions not to be raised in court: No question shall be raised in any court as 
to whether any of the provisions contained in thi

In [15]:
# search for the actual article about PM appointment/formation
council_chunks = [c for c in chunks if "Council of Ministers" in c and ("appoint" in c.lower() or "form" in c.lower())]
print(f"Found {len(council_chunks)} relevant chunks")
for c in council_chunks[:3]:
    print(c)
    print("---")

Found 12 relevant chunks
015, Unofficial English Translation 
 
- 32 - 
PART 7 
Federal Executive 
74. Form of governance: The form of governance of Nepal shall be a multi-party, 
competitive, federal democratic republican parliamentary system based on 
plurality.  
75. Executive Power :  (1) The executive power of Nepal shall rest with the 
Council of Ministers in accordance with this Constitution and law. 
(2) The responsibility of providing general directives, control and 
enforcement regarding the governance system 
---
 and 
enforcement regarding the governance system of Nepal, by adhering to 
this constitution and law, shall rest with the Council of Ministers. 
(3) The entire works relating to the federa l executive of Nepal shall be done 
in the name of the Government of Nepal. 
(4) The decision or Order and related certification of credentials as provided 
for by clause (3) shall be done according to law. 
76. Formation of the Council of Ministers : (1) The President shall appo

In [16]:
answer = generate_answer("How is the Prime Minister of Nepal appointed?", k=5)
print(answer)

Based on the provided context, there is not enough information to fully answer how the Prime Minister of Nepal is appointed. The text mentions in Article 76(1) that "The President shall appoint the parliamentary party l...", but the text cuts off and does not provide the complete process.


In [17]:
answer = generate_answer("How is the Prime Minister of Nepal appointed?", k=8)
print(answer)


Based on the provided context, there is not enough information to fully answer how the Prime Minister of Nepal is appointed. 

The context mentions under Article 76(1) that "The President shall appoint the parliamentary party l..." but the text cuts off before detailing the full appointment process.


## Step 7: Fix chunking — increase chunk size and overlap

Testing revealed a real limitation: our 500-character chunks were cutting
key sentences (like Article 76's PM appointment process) in half, and even
retrieving more chunks (k=8) didn't reliably pull in the continuation.

Fix: increase chunk_size to 1000 and overlap to 200, so fewer sentences
get split, and re-embed + rebuild the FAISS index with the new chunks.

In [18]:
# rebuild with larger chunks and more overlap
chunks = chunk_text(full_text, chunk_size=1000, overlap=200)
print("New number of chunks:", len(chunks))

# re-embed and rebuild index
embeddings = embed_model.encode(chunks, show_progress_bar=True)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print("Number of vectors in new index:", index.ntotal)

# retest the same question
answer = generate_answer("How is the Prime Minister of Nepal appointed?", k=5)
print("\n--- Answer ---")
print(answer)

New number of chunks: 427


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Number of vectors in new index: 427

--- Answer ---
Based on the provided context, the Prime Minister of Nepal is appointed by the **President** under the following provisions:

1. **Party with Highest Number of Members:** The President appoints [a member/leader associated with] the highest number of members in the House of Representatives as the Prime Minister (Clause 3).
2. **Alternative Appointment (Base for Confidence):** If a Prime Minister appointed under Clause (3) fails to receive a vote of confidence within 30 days of appointment, the President shall appoint a member as Prime Minister who produces bases demonstrating that he or she may win the vote of confidence of the House of Representatives (Clause 5).

*Note: Any Prime Minister appointed under these provisions must receive a vote of confidence from the House of Representatives within 30 days of their appointment.*


In [19]:
print(generate_answer("What are the fundamental rights of citizens?", k=3))
print("\n" + "="*80 + "\n")
print(generate_answer("What does the constitution say about the death penalty?", k=3))

Based on the provided context, the fundamental rights mentioned for citizens (as well as persons/women) include:

*   **Right to Equality (Section 18):** All citizens are equal before the law and entitled to equal protection, with no discrimination on grounds such as origin, religion, race, caste, tribe, sex, physical conditions, disability, health condition, matrimonial status, pregnancy, economic condition, language, geographical region, or ideology.
*   **Health and Clean Water Rights:** Citizens have the right to seek basic health care services, must not be deprived of emergency health care, and have the right to access clean water and hygiene.
*   **Right to Food (Section 36):** Citizens have the right to food, the right to be protected from starvation, and the right to food sovereignty as provided for in law.
*   **Right to Housing (Section 37):** Citizens have the right to appropriate housing and protection against unlawful eviction or encroachment on their owned housing.
*   **

In [20]:
answer = generate_answer("How is the Prime Minister of Nepal appointed?", k=8)
print(answer)


Based on the provided context, the text does not contain complete information detailing all the initial procedures for appointing the Prime Minister, as parts of the relevant section are truncated. However, the available context outlines the following details regarding the appointment process:

* **By parliamentary representation:** The text references appointing a Prime Minister associated with the "highest number of members in the House of Representatives" (clause 3).
* **By demonstrating confidence:** If a Prime Minister appointed under clause (3) fails to receive a vote of confidence within 30 days, the President shall appoint a member as Prime Minister who demonstrates that he/she can win a vote of confidence in the House of Representatives (clause 5).
* **Vote of confidence requirement:** Any Prime Minister appointed under these provisions must obtain a vote of confidence from the House of Representatives within 30 days of appointment.


In [21]:
answer = generate_answer("What does the constitution say about the death penalty?", k=6)
print(answer)

Based on the provided context, there is no information about the death penalty.


In [22]:
dignity_chunks = [c for c in chunks if "capital punishment" in c or "live with dignity" in c]
print(f"Found {len(dignity_chunks)} chunks")
for c in dignity_chunks:
    print(c)
    print("---")

Found 1 chunks
al law.  
 
15. Other provisions related to citizensh ip of Nepal : Record keeping of each 
citizen of Nepal along with his/her identity and other provisions regarding 
citizenship of Nepal shall be as provided for by the federal law.  
 
Constitution of Nepal 2015, Unofficial English Translation 
 
- 5 - 
PART 3 
Fundamental Rights and Duties 
16. Right to live with dignity : (1) Each person shall have the rig ht to live with 
dignity. 
(2) No law shall be made for capital punishment. 
17. Right to Freedom : (1) Except as provided for by law no person shall be 
deprived of her/his personal liberty.  
(2) Every citizen shall have the following freedoms:  
(a) freedom of opinion and expression, 
(b) freedom to assemble peacefully and without arms, 
(c) freedom to form political party, 
(d) freedom to form unions and associations, 
(e) freedom to move and reside in any part of Nepal; and 
(f) freedom to engage in any occupation   or be engaged in 
employment, establish and

## Step 8: Diagnose a retrieval miss — "death penalty" query fails despite exact text existing

Testing revealed that asking about "the death penalty" fails to retrieve
the chunk containing "No law shall be made for capital punishment," even
though we confirmed via direct string search that the text exists intact.

Hypothesis: the embedding model compares semantic meaning, and this chunk's
dominant topic is "Right to Freedom" (personal liberty, movement, opinion),
not capital

In [23]:
query_embedding = embed_model.encode(["What does the constitution say about the death penalty?"])
target_embedding = embed_model.encode([dignity_chunks[0]])

import numpy as np
similarity = np.dot(query_embedding[0], target_embedding[0]) / (np.linalg.norm(query_embedding[0]) * np.linalg.norm(target_embedding[0]))
print(f"Similarity score (full question): {similarity:.4f}")

alt_query_embedding = embed_model.encode(["capital punishment"])
alt_similarity = np.dot(alt_query_embedding[0], target_embedding[0]) / (np.linalg.norm(alt_query_embedding[0]) * np.linalg.norm(target_embedding[0]))
print(f"Similarity score ('capital punishment' only): {alt_similarity:.4f}")

Similarity score (full question): 0.2971
Similarity score ('capital punishment' only): 0.1920


## Step 9: Fix — structure-aware chunking by article number

The real issue was chunk-level dilution: short but important clauses (like
capital punishment) get buried inside chunks dominated by unrelated
surrounding content. The fix: split the document by its own structure —
one chunk per numbered article (e.g., "16. Right to live with dignity...")
— rather than fixed character counts. This keeps each legal provision
atomic and undiluted.

In [24]:
import re

# match patterns like "16. Right to live with dignity :" - number, period, space, capital letter
article_pattern = re.compile(r'\n\s*(\d{1,3})\.\s+([A-Z][^:]{2,80}):')

matches = list(article_pattern.finditer(full_text))
print(f"Found {len(matches)} potential article boundaries")

article_chunks = []
for i in range(len(matches)):
    start = matches[i].start()
    end = matches[i+1].start() if i + 1 < len(matches) else len(full_text)
    article_text = full_text[start:end].strip()
    if len(article_text) > 20:  # skip tiny/false matches
        article_chunks.append(article_text)

print(f"Number of article-based chunks: {len(article_chunks)}")
print("\nExample chunk:")
print(article_chunks[10])

Found 285 potential article boundaries
Number of article-based chunks: 285

Example chunk:
11. To be deemed  citizen of Nepal : (1) Persons who have acquired citizenship 
of Nepal at the commencement of this Constitution and persons who are 
eligible to acquire citizenship pursuant to  this Part shall be deemed citizen s 
of Nepal.  
 
(2) At the commencement of this Constitution, the following persons who 
have their permanent domicile in Nepal shall be deemed citizens of Nepal by 
descent: 
 
(a) A person who has acquired the citizenship  of Nepal  by descent 
before the commencement of this Constitution.  
(b) Any person whose father or mother was a citizen of Nepal at the 
birth of such person.  
 
(3) A child of a citizen who has acquired the citizenship of Nepal by birth 
before the commencement of this Constitution, shall acquire the citizenship 
of Nepal by descent after becoming adult if his/her father and mother are 
both citizens of Nepal.  
 
(4) Every  child who is found i

## Step 10: Rebuild the index with article-based chunks and retest

Replace the previous fixed-size chunking with the structure-aware
article chunks, re-embed, rebuild the FAISS index, and retest the
death penalty query that failed under both previous chunking strategies.

In [25]:
chunks = article_chunks  # replace previous chunking with structure-aware version

embeddings = embed_model.encode(chunks, show_progress_bar=True)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print("Number of vectors in index:", index.ntotal)

# retest the failing query
answer = generate_answer("What does the constitution say about the death penalty?", k=3)
print("\n--- Death penalty answer ---")
print(answer)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Number of vectors in index: 285

--- Death penalty answer ---
Based on the provided context, Article 16 (2) states that "No law shall be made for capital punishment."


In [26]:
test_questions = [
    "What are the fundamental rights of citizens?",
    "How is the Prime Minister of Nepal appointed?",
    "What does the constitution say about the death penalty?",
    "What are the qualifications to become Prime Minister of Nepal?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(generate_answer(q, k=3))
    print("\n" + "="*80 + "\n")

Q: What are the fundamental rights of citizens?
Based on the provided context, the fundamental rights specified are:

1. **Right to health care (Section 35):**
   * The right to seek basic health care services from the state and not be deprived of emergency health care.
   * The right to be informed about one's health condition (applies to each person).
   * The right to equal access to health care (applies to each person).
   * The right to access clean water and hygiene.

2. **Right to food (Section 36):**
   * The right to food.
   * The right to be protected from a state of starvation resulting from a lack of foodstuffs.
   * The right to food sovereignty as provided for in law.

3. **Right to housing (Section 37):**
   * The right to appropriate housing.
   * Protection from being evicted from housing owned by the citizen, or having that housing encroached upon, except in accordance with law.

*(Note: The provided context only contains Sections 35–37, so this list reflects only th

In [27]:
print(generate_answer("What are the fundamental rights of citizens?", k=15))
print("\n" + "="*80 + "\n")
print(generate_answer("How is the Prime Minister of Nepal appointed?", k=8))

Based on the provided context, the fundamental rights detailed are:

1. **Right to Health Care (Article 35):**
   * Right of every citizen to seek basic health care services from the state and not be deprived of emergency health care.
   * Right to be informed about one's health condition regarding health care services.
   * Equal access to health care for each person.
   * Right of every citizen to access clean water and hygiene.

2. **Right to Food (Article 36):**
   * Right of each citizen to food.
   * Right to be protected from a state of starvation resulting from lack of food.
   * Right to food sovereignty as provided for in law.

3. **Right to Housing (Article 37):**
   * Right of each citizen to appropriate housing.
   * Protection from being evicted or encroached upon regarding owned housing, except in accordance with law.

4. **Right to Social Justice (Article 42):**
   * Right to employment in state structures based on the principle of inclusion for socially backward, margi

## Step 11: Diagnose why Article 76 (PM appointment) still isn't retrieved

Fundamental rights improved significantly with k=15, but PM appointment
still fails even at k=8, despite Article 76 existing as a clean, complete
chunk. Let's find its exact rank in the full similarity search across all
285 chunks to see how far off retrieval actually is — and rule out the
possibility that the article splitting missed or mangled Article 76 itself.

In [28]:
# find the Article 76 chunk
article_76 = [c for c in chunks if c.strip().startswith("76.")]
print(f"Found {len(article_76)} chunk(s) starting with '76.'")
if article_76:
    print(article_76[0][:300])

# check its similarity rank against the query
query = "How is the Prime Minister of Nepal appointed?"
query_embedding = embed_model.encode([query])
distances, indices = index.search(np.array(query_embedding).astype('float32'), k=285)  # search ALL chunks

# find where article 76's chunk landed in the ranking
target_idx = chunks.index(article_76[0]) if article_76 else None
if target_idx is not None:
    rank = list(indices[0]).index(target_idx)
    print(f"\nArticle 76 chunk rank in results: {rank + 1} out of 285")

Found 1 chunk(s) starting with '76.'
76. Formation of the Council of Ministers : (1) The President shall appoint the 
parliamentary party leader of the political party with the majority in the 
House of Representatives as a Prime Minister, and a Council of Ministers shall 
be formed in his/her chairmanship. 
(2) If there is not a clear

Article 76 chunk rank in results: 49 out of 285


## Step 12: Fix — hybrid search (BM25 keyword search + embedding search)

The core problem: Article 76's title ("Formation of the Council of
Ministers") doesn't semantically match "Prime Minister appointed," even
though the content is exactly relevant. Pure embedding search misses this.

Fix: combine BM25 (classic keyword/term-frequency search, which will match
"Prime Minister" appearing directly in the article body regardless of its
title) with our existing embedding search, then merge and re-rank results
from both. This is called hybrid search — a standard, well-established
technique for exactly this kind of failure mode.

In [29]:
!pip install rank_bm25 -q

from rank_bm25 import BM25Okapi

# tokenize all chunks for BM25 (simple whitespace/lowercase tokenization)
tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def hybrid_retrieve(query, k=5, bm25_weight=0.5):
    # BM25 scores (keyword-based)
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_scores_norm = bm25_scores / (bm25_scores.max() + 1e-8)  # normalize to 0-1

    # embedding scores (semantic) - convert L2 distance to similarity
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k=len(chunks))
    embed_scores = np.zeros(len(chunks))
    max_dist = distances[0].max()
    for rank, idx in enumerate(indices[0]):
        embed_scores[idx] = 1 - (distances[0][rank] / (max_dist + 1e-8))  # normalize, higher = better

    # combine scores
    combined_scores = bm25_weight * bm25_scores_norm + (1 - bm25_weight) * embed_scores
    top_k_idx = np.argsort(combined_scores)[::-1][:k]

    return [chunks[i] for i in top_k_idx]

# test on the failing query
def generate_answer_hybrid(query, k=5):
    retrieved_chunks = hybrid_retrieve(query, k=k)
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""Answer the question using ONLY the context below, which is from the Constitution of Nepal (2015, as amended). If the context doesn't contain enough information to answer, say so clearly rather than guessing.

Context:
{context}

Question: {query}

Answer:"""

    response = client.models.generate_content(model="gemini-3.6-flash", contents=prompt)
    return response.text

print(generate_answer_hybrid("How is the Prime Minister of Nepal appointed?", k=5))

Based on the provided context (primarily Article 76, Article 100, and Article 298 of the Constitution of Nepal), the Prime Minister is appointed by the President through the following procedures:

1. **Majority Party Leader (Article 76(1)):** The President appoints the parliamentary party leader of the political party that holds a majority in the House of Representatives.

2. **Coalition Majority Member (Article 76(2)):** If no single party has a clear majority, the President appoints the member of the House of Representatives who can command a majority with the support of two or more political parties represented in the House.

3. **Leader of the Largest Party (Article 76(3)):** If a Prime Minister cannot be appointed under clause (2) within 30 days of the election results, or if the Prime Minister appointed under clause (2) fails to win a vote of confidence, the President appoints the leader of the political party with the highest number of members in the House of Representatives.

4

In [31]:
import time

test_questions = [
    "What are the fundamental rights of citizens?",
    "How is the Prime Minister of Nepal appointed?",
    "What does the constitution say about the death penalty?",
    "What are the qualifications to become Prime Minister of Nepal?",
]

for q in test_questions:
    print(f"Q: {q}")
    try:
        print(generate_answer_hybrid(q, k=5))
    except Exception as e:
        print(f"Rate limited, waiting 45s and retrying...")
        time.sleep(45)
        print(generate_answer_hybrid(q, k=5))
    print("\n" + "="*80 + "\n")

Q: What are the fundamental rights of citizens?
Rate limited, waiting 45s and retrying...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 10.723502016s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},